# Federated Learning — FedAvg on Tomato Leaf Disease Dataset

ResNet18 | SGD (lr=0.001, momentum=0.9) | batch=32 | 5 local epochs | 10 global rounds | 5 clients | Dirichlet non-IID (α=0.5)

Dataset: https://www.kaggle.com/datasets/kaustubhb999/tomatoleaf (10 classes, ~10,000 train + ~1,000 val images)

This notebook auto-detects the dataset path, partitions the combined train+val pool across 5 simulated clients with a Dirichlet(α=0.5) split, holds out a stratified test set for evaluation, runs FedAvg for 10 rounds, and saves Accuracy/Precision/Recall/F1 to JSON **after every round** plus a resumable checkpoint.

**Before running:** Settings → Accelerator → **GPU T4 x2** (not P100) · Internet → **ON** (needed once, to download ImageNet-pretrained ResNet18 weights).

In [ ]:
import os, json, time, random
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# ---------------- Config ----------------
NUM_CLIENTS   = 5
ALPHA         = 0.5          # Dirichlet concentration
LOCAL_EPOCHS  = 5
GLOBAL_ROUNDS = 10
BATCH_SIZE    = 32
LR            = 0.001
MOMENTUM      = 0.9
IMG_SIZE      = 224
TEST_FRACTION = 0.15         # stratified held-out test set for evaluation
PRETRAINED    = True         # ImageNet-pretrained ResNet18 backbone for the initial global model

OUT_DIR = "/kaggle/working/fedavg_tomatoleaf"
os.makedirs(OUT_DIR, exist_ok=True)
METRICS_PATH = os.path.join(OUT_DIR, "metrics.json")
BEST_PATH    = os.path.join(OUT_DIR, "best_model.pth")
LATEST_PATH  = os.path.join(OUT_DIR, "latest_checkpoint.pth")

# ---------------- Locate dataset (robust to mount-point variants) ----------------
def find_dataset_root(base="/kaggle/input"):
    # Search for a directory that has both a 'train' and a 'val' subfolder,
    # each containing class subfolders. Handles the standard
    # /kaggle/input/tomatoleaf/tomato/{train,val} layout as well as
    # /kaggle/input/datasets/... variants.
    for root, dirs, files in os.walk(base):
        lower_dirs = [d.lower() for d in dirs]
        if "train" in lower_dirs and "val" in lower_dirs:
            train_dir = os.path.join(root, [d for d in dirs if d.lower() == "train"][0])
            val_dir   = os.path.join(root, [d for d in dirs if d.lower() == "val"][0])
            if any(os.path.isdir(os.path.join(train_dir, c)) for c in os.listdir(train_dir)):
                return train_dir, val_dir
    raise FileNotFoundError(
        "Could not find train/val folders under /kaggle/input. "
        "Make sure the 'tomatoleaf' dataset is attached to this notebook."
    )

TRAIN_DIR, VAL_DIR = find_dataset_root()
print("Train dir:", TRAIN_DIR)
print("Val dir:  ", VAL_DIR)

CLASSES = sorted(os.listdir(TRAIN_DIR))
NUM_CLASSES = len(CLASSES)
print(f"Found {NUM_CLASSES} classes:", CLASSES)

In [ ]:
# ---------------- Collect all image paths + labels ----------------
def collect_samples(directory, classes):
    samples = []
    for label_idx, cls in enumerate(classes):
        cls_dir = os.path.join(directory, cls)
        for fname in os.listdir(cls_dir):
            if fname.lower().endswith((".jpg", ".jpeg", ".png", ".bmp")):
                samples.append((os.path.join(cls_dir, fname), label_idx))
    return samples

all_samples = collect_samples(TRAIN_DIR, CLASSES) + collect_samples(VAL_DIR, CLASSES)
print("Total images found:", len(all_samples))

paths  = np.array([s[0] for s in all_samples])
labels = np.array([s[1] for s in all_samples])

# Stratified global test split (held out, never seen by any client)
train_idx, test_idx = train_test_split(
    np.arange(len(paths)), test_size=TEST_FRACTION, stratify=labels, random_state=SEED
)
test_paths, test_labels = paths[test_idx], labels[test_idx]
pool_paths, pool_labels = paths[train_idx], labels[train_idx]
print(f"Held-out test set: {len(test_idx)} images | Client pool: {len(train_idx)} images")

# ---------------- Dirichlet non-IID partition across clients ----------------
def dirichlet_partition(labels, num_clients, alpha, seed=SEED):
    rng = np.random.RandomState(seed)
    num_classes = labels.max() + 1
    client_indices = [[] for _ in range(num_clients)]
    for c in range(num_classes):
        idx_c = np.where(labels == c)[0]
        rng.shuffle(idx_c)
        proportions = rng.dirichlet(alpha=[alpha] * num_clients)
        split_points = (np.cumsum(proportions) * len(idx_c)).astype(int)[:-1]
        splits = np.split(idx_c, split_points)
        for client_id, split in enumerate(splits):
            client_indices[client_id].extend(split.tolist())
    for c in range(num_clients):
        rng.shuffle(client_indices[c])
    return client_indices

client_indices = dirichlet_partition(pool_labels, NUM_CLIENTS, ALPHA)
for i, idx in enumerate(client_indices):
    cls_counts = np.bincount(pool_labels[idx], minlength=NUM_CLASSES)
    print(f"Client {i}: {len(idx)} images | class distribution: {cls_counts.tolist()}")

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class LeafDataset(Dataset):
    def __init__(self, paths, labels, transform):
        self.paths = paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        img = self.transform(img)
        label = int(self.labels[idx])
        return img, label

client_loaders = []
for idx in client_indices:
    ds = LeafDataset(pool_paths[idx], pool_labels[idx], train_transform)
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True, drop_last=False)
    client_loaders.append(loader)

test_dataset = LeafDataset(test_paths, test_labels, eval_transform)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print("Client loaders ready:", [len(l.dataset) for l in client_loaders])
print("Test loader ready:", len(test_loader.dataset))

In [ ]:
def build_model(num_classes=NUM_CLASSES, pretrained=False):
    if pretrained:
        model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    else:
        model = models.resnet18(weights=None)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model.to(device)

In [ ]:
def local_train(global_state_dict, loader, epochs=LOCAL_EPOCHS):
    # Train a fresh copy of the global model on one client's local data for
    # `epochs` local epochs. Returns the resulting state_dict (on CPU), the
    # number of samples used, and the average training loss.
    model = build_model(pretrained=False)   # weights overwritten by global_state_dict below
    model.load_state_dict(global_state_dict)
    model.train()

    optimizer = optim.SGD(model.parameters(), lr=LR, momentum=MOMENTUM)
    criterion = nn.CrossEntropyLoss()

    running_loss = 0.0
    n_batches = 0
    for epoch in range(epochs):
        for images, targets in loader:
            images = images.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            n_batches += 1

    avg_loss = running_loss / max(n_batches, 1)
    cpu_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    del model
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return cpu_state, len(loader.dataset), avg_loss


def fedavg_aggregate(client_states, client_sizes):
    # Weighted average of client state_dicts, weighted by local dataset size
    # (standard FedAvg aggregation).
    total = sum(client_sizes)
    avg_state = {}
    for key in client_states[0].keys():
        if client_states[0][key].dtype.is_floating_point:
            stacked = torch.stack(
                [client_states[i][key].float() * (client_sizes[i] / total) for i in range(len(client_states))],
                dim=0,
            )
            avg_state[key] = stacked.sum(dim=0).to(client_states[0][key].dtype)
        else:
            # Non-float buffers (e.g. BatchNorm num_batches_tracked) -> copy from the largest client
            largest = int(np.argmax(client_sizes))
            avg_state[key] = client_states[largest][key].clone()
    return avg_state


@torch.no_grad()
def evaluate(model_state_dict, loader):
    model = build_model(pretrained=False)
    model.load_state_dict(model_state_dict)
    model.eval()

    all_preds, all_targets = [], []
    for images, targets in loader:
        images = images.to(device, non_blocking=True)
        outputs = model(images)
        preds = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds.tolist())
        all_targets.extend(targets.numpy().tolist())

    del model
    if device.type == "cuda":
        torch.cuda.empty_cache()

    acc  = accuracy_score(all_targets, all_preds)
    prec = precision_score(all_targets, all_preds, average="macro", zero_division=0)
    rec  = recall_score(all_targets, all_preds, average="macro", zero_division=0)
    f1   = f1_score(all_targets, all_preds, average="macro", zero_division=0)
    return acc, prec, rec, f1

In [ ]:
# ---------------- Resume support ----------------
if os.path.exists(METRICS_PATH):
    with open(METRICS_PATH, "r") as f:
        metrics_history = json.load(f)
else:
    metrics_history = []

start_round = len(metrics_history) + 1
best_f1 = max((m["f1"] for m in metrics_history), default=-1.0)

if os.path.exists(LATEST_PATH):
    checkpoint = torch.load(LATEST_PATH, map_location=device)
    global_state = checkpoint["model_state"]
    print(f"Resuming from saved checkpoint -> starting at round {start_round}/{GLOBAL_ROUNDS} "
          f"(best F1 so far: {best_f1:.4f})")
else:
    global_state = {k: v.detach().cpu().clone() for k, v in build_model(pretrained=PRETRAINED).state_dict().items()}
    print(f"No checkpoint found -> starting fresh from round 1/{GLOBAL_ROUNDS}")

# ---------------- FedAvg main loop ----------------
for rnd in range(start_round, GLOBAL_ROUNDS + 1):
    t0 = time.time()
    client_states, client_sizes, client_losses = [], [], []

    for client_id, loader in enumerate(client_loaders):
        state, n_samples, avg_loss = local_train(global_state, loader, epochs=LOCAL_EPOCHS)
        client_states.append(state)
        client_sizes.append(n_samples)
        client_losses.append(avg_loss)
        print(f"  Round {rnd} | Client {client_id} | n={n_samples} | local_loss={avg_loss:.4f}")

    global_state = fedavg_aggregate(client_states, client_sizes)

    acc, prec, rec, f1 = evaluate(global_state, test_loader)
    elapsed = time.time() - t0

    record = {
        "round": rnd,
        "avg_client_loss": float(np.mean(client_losses)),
        "accuracy": float(acc),
        "precision": float(prec),
        "recall": float(rec),
        "f1": float(f1),
        "elapsed_sec": round(elapsed, 1),
    }
    metrics_history.append(record)

    # Save metrics immediately, every round
    with open(METRICS_PATH, "w") as f:
        json.dump(metrics_history, f, indent=2)

    # Save latest checkpoint immediately, every round (for resume)
    torch.save({"round": rnd, "model_state": global_state}, LATEST_PATH)

    # Save best checkpoint only
    if f1 > best_f1:
        best_f1 = f1
        torch.save({"round": rnd, "model_state": global_state, "f1": f1}, BEST_PATH)
        print(f"  -> New best model saved (F1={f1:.4f})")

    print(f"Round {rnd}/{GLOBAL_ROUNDS} done in {elapsed:.1f}s | "
          f"Acc={acc:.4f} Prec={prec:.4f} Rec={rec:.4f} F1={f1:.4f}")
    print("-" * 70)

print("FedAvg training complete.")
print("Metrics file:", METRICS_PATH)
print("Best model: ", BEST_PATH, f"(F1={best_f1:.4f})")

In [ ]:
import pandas as pd
df = pd.DataFrame(metrics_history)
print(df.to_string(index=False))